# ¿Y si el grupo 2 se jugara 10.000 veces?

**Línea de pase** · Ángel Marqueta Alonso

Simulación Monte Carlo del resto de la temporada del grupo 2 de Primera Federación. El cuaderno llama a las funciones de `codigo/` para que dé exactamente lo mismo que el programa.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
RAIZ = Path.cwd().parent if Path.cwd().name == "cuadernos" else Path.cwd()
sys.path.insert(0, str(RAIZ / "codigo"))
from datos import leer_resultados, comprobar_contra_clasificacion, partidos_pendientes, matriz_goles
from modelo import ajustar, elegir_penalizacion, probabilidades_1x2, tasas
from simulacion import simular, resumir

## 1. Resultados y comprobación

Solo hacen falta los resultados. Antes de nada compruebo que reproducen la clasificación publicada: si falta un partido o hay un marcador mal apuntado, la comprobación falla.

In [2]:
partidos = leer_resultados()
jornada = comprobar_contra_clasificacion(partidos)
equipos, pendientes = partidos_pendientes(partidos)
print(f"Tras la jornada {jornada}: {len(partidos)} partidos jugados, {len(pendientes)} pendientes")
partidos.tail()

Tras la jornada 7: 70 partidos jugados, 310 pendientes


,jornada,fecha,local,visitante,goles_local,goles_visitante
65,7,2026-10-11,Antequera,Cartagena,1,1
66,7,2026-10-11,Atlético Madrileño,Real Zaragoza,4,4
67,7,2026-10-11,CE Europa,Algeciras,2,1
68,7,2026-10-11,Rayo Majadahonda,Real Murcia,0,1
69,7,2026-10-12,Hércules,Sant Andreu,1,0


## 2. Cuánto creer a siete jornadas

El modelo acerca las fuerzas de los equipos a la media del grupo. La validación cruzada dice cuánto: para cada penalización, ajusto sin una jornada y mido lo bien que predigo esa jornada (log-verosimilitud media por partido; más alto es mejor).

In [3]:
penalizacion, puntuaciones = elegir_penalizacion(partidos, equipos)
print("Penalización elegida:", penalizacion)
pd.Series(puntuaciones, name="log-verosimilitud").round(4)

Penalización elegida: 32


0.5    -3.1961
1.0    -3.1516
2.0    -3.0936
4.0    -3.0337
8.0    -2.9879
16.0   -2.9654
32.0   -2.9618
64.0   -2.9667
Name: log-verosimilitud, dtype: float64

La mejor opción es una penalización alta: con siete jornadas, los resultados todavía dicen poco de la fuerza real de cada equipo. Es una lección en sí misma: la clasificación de octubre se parece poco a la de mayo.

## 3. Modelo y probabilidades de un partido

In [4]:
modelo = ajustar(partidos, equipos, penalizacion)
print(f"Ventaja de jugar en casa: el local marca {np.exp(modelo['theta'][1]):.2f} veces lo que marcaría fuera")
propios = pendientes[(pendientes.local == "Real Zaragoza") | (pendientes.visitante == "Real Zaragoza")].head(4)
a, b = tasas(modelo, propios.local, propios.visitante)
p1, px, p2 = probabilidades_1x2(a, b)
propios.assign(victoria_local=p1.round(2), empate=px.round(2), victoria_visitante=p2.round(2))

Ventaja de jugar en casa: el local marca 1.43 veces lo que marcaría fuera


,local,visitante,victoria_local,empate,victoria_visitante
11,Alcorcón,Real Zaragoza,0.50,0.24,0.26
27,Algeciras,Real Zaragoza,0.42,0.24,0.34
43,Antequera,Real Zaragoza,0.47,0.24,0.29
73,CD Teruel,Real Zaragoza,0.37,0.26,0.37


## 4. Simulación

10.000 temporadas. En cada una saco unas fuerzas dentro de su incertidumbre, juego los partidos pendientes y ordeno la tabla con el reglamento.

In [5]:
puestos, puntos = simular(modelo, matriz_goles(partidos, equipos), pendientes, n=10_000)
tabla, distribucion = resumir(puestos, puntos, equipos)
assert np.allclose(distribucion.sum(axis=0), 1) and np.allclose(distribucion.sum(axis=1), 1)
tabla[["equipo", "puntos_medios", "ascenso_directo", "playoff", "descenso"]].round(3)

,equipo,puntos_medios,ascenso_directo,playoff,descenso
0,Real Zaragoza,63.349,0.203,0.427,0.024
1,Villarreal B,61.915,0.160,0.414,0.032
2,Real Murcia,60.696,0.132,0.389,0.040
3,Alcorcón,59.498,0.111,0.369,0.052
4,Cartagena,58.673,0.088,0.353,0.062
5,Real Jaén,57.263,0.073,0.306,0.083
6,Antequera,54.913,0.041,0.244,0.133
7,Hércules,53.818,0.033,0.213,0.147
8,Atlético Madrileño,53.614,0.033,0.221,0.170
9,UD Ibiza,52.841,0.028,0.193,0.182


## 5. Lectura

- El Real Zaragoza es el equipo con más opciones, pero lejos de tenerlo decidido.
- La diferencia entre los seis primeros es pequeña: siete jornadas no separan a los equipos tanto como parece en la tabla.

**Límites.** No sabe de lesiones ni fichajes; los goles se suponen independientes; el juego limpio y el desempate en campo neutral se sortean.